# 🎬 MiniMax-H3 Colab `A100 (40GB)` 自动化渲染工作台

**架构理念（Zero-Local-Disk + Zero-Idle-CU）**：
- **本地控制面**：AI Agent 仅负责编剧、分镜与编译几 KB 的任务单（0 算力消耗、0 本地媒体囤积）。
- **Google Drive 云存储面**：所有模型权重 (`cache/`)、参考图/音视频 (`assets/`) 与生成成片 (`outputs/`) 100% 存储在 Google Drive。
- **Colab A100 算力面**：读取 `inbox/` 极速批量出片，刷盘后自动调用 `runtime.unassign()` **立即释放 GPU 停止计费**。

### 使用前检查
顶部菜单：**代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type)**，硬件加速器选择 **`A100 GPU`**（标准 40GB 档，约 `5.37 CU/h`）。

In [ ]:
#@title 1️⃣ 挂载 Google Drive 并就绪 Pipeline 工程目录
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

WORKSPACE_DRIVE = Path('/content/drive/MyDrive/MiniMaxH3_Workspace')
PIPELINE_DIR = WORKSPACE_DRIVE / 'pipeline'
WORKSPACE_DRIVE.mkdir(parents=True, exist_ok=True)

if not (PIPELINE_DIR / 'scripts' / 'colab_bootstrap.sh').is_file():
    print('📦 云盘尚未同步工程脚本，正在从公开仓库 flynn-os/MiniMaxH3_pipeline 自动拉取...')
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/flynn-os/MiniMaxH3_pipeline.git',
        str(PIPELINE_DIR),
    ], check=True)

print(f'✅ Google Drive 云端工作区: {WORKSPACE_DRIVE}')
print(f'✅ Pipeline 脚本目录已就绪: {PIPELINE_DIR}')

In [ ]:
#@title 2️⃣ 一键初始化 WanGP 与持久化云盘缓存 (首次约 2-3 分钟，后续秒过)
!WORKSPACE_DRIVE="/content/drive/MyDrive/MiniMaxH3_Workspace" \
  WANGP_DIR="/content/WanGP" \
  bash "/content/drive/MyDrive/MiniMaxH3_Workspace/pipeline/scripts/colab_bootstrap.sh"

In [ ]:
#@title 3️⃣ 【主力省点模式】批量渲染队列并在完成后自动断开 A100
#@markdown - `JOB_FILE` 留空则自动扫描并执行 `MiniMaxH3_Workspace/inbox/` 下所有待跑任务（跑完自动归档至 `inbox/processed/`）。
#@markdown - 若 `inbox/` 为空且勾选 `RUN_BUILTIN_DEMO_IF_EMPTY`，将自动运行内置雨夜拉面样片。
JOB_FILE = ""  #@param {type:"string"}
RUN_BUILTIN_DEMO_IF_EMPTY = True  #@param {type:"boolean"}
AUTO_DISCONNECT = True  #@param {type:"boolean"}
GRACE_SECONDS = 15  #@param {type:"integer"}

import subprocess, sys
from pathlib import Path

ws = Path('/content/drive/MyDrive/MiniMaxH3_Workspace')
inbox = ws / 'inbox'
runner = ws / 'pipeline' / 'scripts' / 'colab_runner.py'

cmd = [
    sys.executable, str(runner),
    '--mode', 'batch',
    '--grace-seconds', str(GRACE_SECONDS),
    '--auto-disconnect' if AUTO_DISCONNECT else '--no-auto-disconnect',
]

if JOB_FILE.strip():
    cmd += ['--job', JOB_FILE.strip()]
elif RUN_BUILTIN_DEMO_IF_EMPTY:
    pending = list(inbox.glob('*.wangp.json')) + list(inbox.glob('*.bundle.zip')) if inbox.is_dir() else []
    if not pending:
        demo_job = ws / 'pipeline' / 'jobs' / 'demo_cyberpunk_ramen.wangp.json'
        print(f'ℹ️ inbox/ 为空，自动载入内置样片队列: {demo_job}')
        cmd += ['--job', str(demo_job)]

subprocess.run(cmd, check=False)

In [ ]:
#@title 4️⃣ 【可选远程交互模式】启动 MCP Server (供本地 AI 实时遥控) 或 WebUI
#@markdown ⚠️ **注意**：此模式会保持长连接等待外部调用，不会自动断连，调试完毕请务必手动断开运行时！
SERVER_MODE = "mcp"  #@param ["mcp", "webui"]
ACCESS_PASSWORD = ""  #@param {type:"string"}

import subprocess, sys

runner = '/content/drive/MyDrive/MiniMaxH3_Workspace/pipeline/scripts/colab_runner.py'
cmd = [sys.executable, runner, '--mode', SERVER_MODE]
if ACCESS_PASSWORD.strip():
    cmd += ['--password', ACCESS_PASSWORD.strip()]

subprocess.run(cmd, check=False)